# Components

Description: compute components of the entailment score given a model and dataset.

---

## Boilerplate

In [1]:
import os
import sys
import json

from pathlib import Path


COLAB_ROOT_PATH = "/content"
IS_COLAB = os.path.exists(COLAB_ROOT_PATH)

if IS_COLAB:
  # Working on Google Colab
  from google.colab import drive

  # Mount Google Drive
  DRIVE_PATH = os.path.join(COLAB_ROOT_PATH, "drive")
  drive.flush_and_unmount()
  drive.mount(DRIVE_PATH)

  # Load config
  with open(os.path.join(DRIVE_PATH, "MyDrive", "Colab", "config.json"), "r") as f:
    config = json.load(f)

  # Set up Git credentials
  GIT_USER_NAME = config["GIT_USER_NAME"]
  GIT_TOKEN = config["GIT_TOKEN"]
  GIT_USER_EMAIL = config["GIT_USER_EMAIL"]

  !git config --global user.email {GIT_USER_EMAIL}
  !git config --global user.name {GIT_USER_NAME}

  # Set up project paths
  GIT_OWNER = "haelai77"
  GIT_REPOSITORY = "COMP0087-Statistical-Natural-Language-Processing"
  STORAGE_PATH = os.path.join(DRIVE_PATH, "MyDrive", config["DRIVE_PATH"], "Colab")
  ROOT_PATH = os.path.join(COLAB_ROOT_PATH, GIT_REPOSITORY)

  # Clone repo
  GIT_PATH = f"https://{GIT_TOKEN}@github.com/{GIT_OWNER}/{GIT_REPOSITORY}.git"

  if not os.path.exists(ROOT_PATH):
    !git clone "{GIT_PATH}" "{ROOT_PATH}"
  else:
    print(f"Git repo already cloned at {ROOT_PATH}")

  %pip install -e {ROOT_PATH}

else:
  # Working on local machine
  # Get the absolute path of the current file
  current_path = Path().resolve()

  # Traverse upwards to find the directory containing "comp0087"
  ROOT_PATH = None
  for parent in current_path.parents:
      if "comp0087" in parent.name.lower():  # Match folder name "comp0087" (case-insensitive)
          ROOT_PATH = parent.resolve()
          break

  # If found, print the root path; otherwise, raise an error
  if not ROOT_PATH:
    raise FileNotFoundError("Directory with name 'comp0087...' not found.")

  # Set the storage path to the root path
  STORAGE_PATH = ROOT_PATH

# Data and output paths
DATA_PATH = os.path.join(STORAGE_PATH, "data")
OUTPUT_PATH = os.path.join(STORAGE_PATH, "output")
MODEL_PATH = os.path.join(STORAGE_PATH, "models")

if not os.path.exists(DATA_PATH):
  # Create if does not exist
  os.makedirs(DATA_PATH)
  print(f"Created data directory at {DATA_PATH}")

if not os.path.exists(OUTPUT_PATH):
  # Create if does not exist
  os.makedirs(OUTPUT_PATH)
  print(f"Created output directory at {OUTPUT_PATH}")

if not os.path.exists(MODEL_PATH):
  # Create if does not exist
  os.makedirs(MODEL_PATH)
  print(f"Created model directory at {MODEL_PATH}")

# Add root path to sys.path
sys.path.append(ROOT_PATH)

print("="*50)
print(f"Runtime: {'Google Colab' if IS_COLAB else 'local machine'}")
print(f"ROOT_PATH: {ROOT_PATH}")
print(f"STORAGE_PATH: {STORAGE_PATH}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")
print(f"MODEL_PATH: {MODEL_PATH}")
print("="*50)

Mounted at /content/drive
Git repo already cloned at /content/COMP0087-Statistical-Natural-Language-Processing
Obtaining file:///content/COMP0087-Statistical-Natural-Language-Processing
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for snlp (pyproject.toml) ... done
  Created wheel for snlp: filename=snlp-0.1.0-py3-none-any.whl size=2506 sha256=0bcce73c5d5fb13bf1fc1ac7f2bc3bc77364114ebd1bd37d963132fa49ad5b2f
  Stored in directory: /tmp/pip-ephem-wheel-cache-sgh770mf/wheels/eb/b0/ba/2ca2d3cd2b512cc1535303455d897f4ca799972f2519d52e5a
Successfully built snlp
  Attempting uninstall: snlp
    Found existing installation: snlp 0.1.0
    Uninstalling snlp-0.1.0:
      Successfully uninstalled snlp-0.1.0
Runtime: Google Colab
ROOT_PATH: /content/COMP0087-Statistical-Natura

---

## Content

### Imports

In [2]:
# Start with imports at the top
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

### Load Models

In [3]:
# # Qwen2.5-0.5B
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B", torch_dtype="auto", device_map="auto", cache_dir=MODEL_PATH
    )
base_tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B", torch_dtype="auto", device_map="auto", cache_dir=MODEL_PATH
    )

# Qwen2.5-0.5B-Instruct
# instruct_model = AutoModelForCausalLM.from_pretrained(
#     "Qwen/Qwen2.5-0.5B-Instruct", torch_dtype="auto", device_map="auto", cache_dir=MODEL_PATH
#     )
# instruct_tokenizer = AutoTokenizer.from_pretrained(
#     "Qwen/Qwen2.5-0.5B-Instruct", torch_dtype="auto", device_map="auto", cache_dir=MODEL_PATH
#     )

### Your Code

In [7]:
# Your main code here
df = pd.read_csv(os.path.join(DATA_PATH, "comVE-mock.csv"))

with open(os.path.join(DATA_PATH, "comVE-template.txt"), "r") as f:
  prompt_template = f.read()

display(df.head())

,id,sentence_0,sentence_1,true_class,pred_class,0_prob,1_prob,intervention,tvd,explanation
0,0,It is easy to find seashells in the forest,It is easy to find seashells by the ocean,0,0,0.991,0.009,NaN,0.02,seashells are found by the ocean
1,0,It is easy to find seashells in the forest,It is easy to find seashells by the gloomy ocean,0,0,0.976,0.024,gloomy,0.02,seashells are found by the ocean
2,1,cars are the only way to get around,cars are a useful mode of transportation,0,0,0.913,0.086,NaN,0.42,There are many ways to get around such as buse...
3,1,cars are the only way to get around,Grey cars are a useful mode of transportation,0,1,0.496,0.503,Grey,0.45,Grey cars are not the only way to get around
4,2,The rotary phone recorded me.,The cell phone recorded me.,0,0,0.981,0.019,NaN,0.00,The rotary phone can't record.


In [8]:
def generate_prompt(prompt_template, placeholders):
    for placeholder, value in placeholders.items():
        prompt_template = prompt_template.replace(f"{{{{{placeholder}}}}}", value)
    return prompt_template

def inference(prompt, model, tokenizer):
  # encode the input
    model_inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    # generate the response
    # outputs has attributes: sequences, scores
    # sequences: the generated token ids in format Tensor[[id1, id2, ...]]
    # scores: the logits of each token at each position in format tuple(Tensor[[logit1, logit2, ...]])
    outputs = model.generate(**model_inputs, max_new_tokens=50, return_dict_in_generate=True, output_scores=True)

    # extract the generated tokens only
    input_length = model_inputs.input_ids.shape[1]
    generated_ids = outputs.sequences[:, input_length:]

    print("\nResponse")
    print("="*70)
    # decode the response
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)
    print("="*70)

    # extract the classification label
    # assumes the label is the last word on the first line of the response
    label = response.splitlines()[0].split(" ")[-1].strip()

    # extract all the text after "explanation: " and before "text:"
    explanation_section = response.lower().split("explanation: ")[-1]
    if "text:" in explanation_section:
        explanation = explanation_section.split("text:")[0].strip()
    else:
        explanation = explanation_section.strip()

    print(label, explanation)
    # compute the probabilities of all possible labels
    # cls_probs = processor.get_cls_probs(generated_ids, outputs.scores, tokenizer)

    # return label, cls_probs, explanation

In [9]:
prompt = generate_prompt(prompt_template, {
    "SENTENCE_0": df.iloc[0]["sentence_0"],
    "SENTENCE_1": df.iloc[0]["sentence_1"]
})

print(prompt)

The following are examples from a dataset. Each example consists of a pair of sentences, "SENTENCE 0" and "SENTENCE 1". One of these sentences violates common sense. Each pair of these is labeled with "FALSE SENTENCE", followed by the label of the false sentence, 0 or 1. "EXPLANATION" explains why sentence is chosen.

SENTENCE 0: You can use a holding bay to store an item
SENTENCE 1: You can use a holding bay to delete an item
FALSE SENTENCE: 1
EXPLANATION: Deleting items is not a holding bay function

SENTENCE 0: Rainbow has five colors
SENTENCE 1: Rainbow has seven colors
FALSE SENTENCE: 0
EXPLANATION: The seven colors of the rainbow are red, orange, yellow, green, blue, blue, and purple

SENTENCE 0: You are likely to find a cat in ocean
SENTENCE 1: You are likely to find a shark in ocean
FALSE SENTENCE: 0
EXPLANATION: Cats do not feed on ocean lives

SENTENCE 0: The caterpillar eats the rose bud
SENTENCE 1: Roses buds eat caterpillars
FALSE SENTENCE: 1
EXPLANATION: Caterpillars have

In [ ]:
inference(prompt, base_model, base_tokenizer)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
